# Pandas进阶

本篇对应课程资料 `06_Pandas进阶`，按讲义与视频顺序整理。重点掌握 DataFrame 的统计与自定义运算、常见文件读写、增删改查、缺失值处理、数据合并、分组分析以及交叉表和透视表。

> 说明：本笔记中的代码用于学习和复习，均放在 Markdown 代码块中；运行前请根据本机数据文件路径、数据库账号和数据表名进行调整。

## 一、进阶内容总览

Pandas 的典型分析流程可以抽象为：

1. 读取 CSV、数据库或 JSON 数据；
2. 检查索引、列名、数据类型和缺失值；
3. 通过布尔条件、`query` 或 `isin` 筛选数据；
4. 使用统计函数、`groupby`、交叉表和透视表提取结论；
5. 用 `concat` 或 `merge` 合并数据；
6. 清洗后保存为文件或写回数据库。

进阶操作的核心是：先明确数据的行粒度、连接键和统计口径，再选择对应 API。

## 二、DataFrame 的运算与筛选

### 2.1 算术运算

Series 或 DataFrame 的算术运算默认按索引对齐。`add`、`sub` 等方法适合表达“整列加上一个值”这类操作。

```python
import pandas as pd

data = pd.read_csv("./data/stock_day.csv")

# 开盘价整体加 1
open_plus_one = data["open"].add(1)

# 收盘价减去 1
close_minus_one = data["close"].sub(1)
```

直接使用 `+`、`-` 也可以，但方法形式更容易继续传入参数并表达意图。

### 2.2 布尔筛选、`query` 与 `isin`

比较运算会返回由 `True` 和 `False` 组成的布尔 Series，再用它筛选行。组合条件时使用 `&`、`|`、`~`，每个条件建议加括号。

```python
# 开盘价大于 23
high_open = data[data["open"] > 23]

# 开盘价位于 (23, 24) 区间
part = data[(data["open"] > 23) & (data["open"] < 24)]

# query 用字符串表达筛选条件
part = data.query("open > 23 and open < 24")

# 筛选指定的开盘价
selected = data[data["open"].isin([23.53, 23.85])]
```

`query` 可读性较好，列名含空格或特殊字符时需要使用反引号，例如 ``df.query('`column name` > 0')``。

### 2.3 描述性统计

`describe()` 一次性给出数值列的数量、均值、标准差、最小值、四分位数和最大值。单个统计函数可以指定轴：`axis=0` 按列统计，`axis=1` 按行统计。

| 方法 | 作用 |
| --- | --- |
| `count` | 非缺失值数量 |
| `sum` | 求和 |
| `mean` | 平均值 |
| `median` | 中位数 |
| `min` / `max` | 最小值 / 最大值 |
| `std` / `var` | 标准差 / 方差 |
| `idxmin` / `idxmax` | 最小值 / 最大值对应的索引 |
| `mode` | 众数 |

```python
summary = data.describe()
column_max = data.max(axis=0)
column_min_index = data.idxmin(axis=0)
```

### 2.4 累计统计

累计函数按当前顺序逐行计算，常用于观察趋势。使用前要先确认索引顺序是否符合时间顺序。

```python
data = data.sort_index()
rise = data["p_change"]

cumulative_sum = rise.cumsum()
cumulative_max = rise.cummax()
cumulative_min = rise.cummin()
cumulative_product = rise.cumprod()

# 可视化累计结果
import matplotlib.pyplot as plt
cumulative_sum.plot()
plt.show()
```

### 2.5 `apply` 自定义运算

`apply(func, axis=0)` 会将函数应用到每一列；`axis=1` 则逐行应用。自定义函数应明确输入是一列 Series 还是一行 Series，以及返回标量还是序列。

```python
# 分别计算 open、close 两列的极差
range_by_column = data[["open", "close"]].apply(
    lambda series: series.max() - series.min(),
    axis=0,
)

# 逐行计算最高价与最低价的差值
data["high_low_range"] = data.apply(
    lambda row: row["high"] - row["low"],
    axis=1,
)
```

优先使用向量化表达式；只有当现成方法难以表达业务规则时再使用 `apply`。

## 三、文件读取与存储

### 3.1 CSV 与 TSV

CSV 以分隔符组织表格数据，TSV 只是把分隔符换成制表符。读取时常用 `sep` 和 `usecols`；写出时重点关注是否保存索引、编码以及覆盖还是追加。

```python
# CSV，默认逗号分隔
df = pd.read_csv("./data/stock_day.csv", usecols=["open", "close"])

# TSV，使用制表符分隔
tsv = pd.read_csv("./data/example.tsv", sep="\t")

# 只保存需要的列，并且不把 DataFrame 索引写成一列
df.head(10).to_csv(
    "./data/test.csv",
    columns=["open"],
    index=False,
    encoding="utf-8-sig",
)
```

如果读回文件后出现 `Unnamed: 0`，通常是之前把索引写进文件造成的；读取时可用 `index_col=0`，或重新写出时设置 `index=False`。

### 3.2 读写 SQL 表

Pandas 通常通过 SQLAlchemy 创建数据库引擎，再调用 `to_sql` 和 `read_sql`。数据库密码应放在环境变量或安全配置中，不要直接提交到代码仓库。

```python
from sqlalchemy import create_engine

engine = create_engine(
    "mysql+pymysql://用户名:密码@127.0.0.1:3306/test?charset=utf8"
)

# 写入数据库；表不存在时创建，存在时追加
df.to_sql(
    "test_pdtosql",
    engine,
    index=False,
    if_exists="append",
)

# 读取整张表
all_rows = pd.read_sql("test_pdtosql", engine)

# 用 SQL 查询需要的列
part_rows = pd.read_sql(
    "SELECT name, AKA FROM test_pdtosql",
    engine,
)
```

常见参数：`if_exists='fail'` 遇到已有表时报错，`replace` 覆盖表，`append` 追加数据。

### 3.3 JSON 的 `orient` 与 `lines`

JSON 读写时要同时考虑对象结构和是否按行存储。新闻数据等“每行一个 JSON 对象”的文件，应使用 `lines=True` 和 `orient="records"`。

| `orient` | 结构特点 |
| --- | --- |
| `split` | `index`、`columns`、`data` 分开保存 |
| `records` | 每条记录是一个字典 |
| `index` | 索引映射到列值字典 |
| `columns` | 列映射到索引和值，常见默认结构 |
| `values` | 只保存值数组 |

```python
# 每行一个 JSON 对象
news = pd.read_json(
    "./data/Sarcasm_Headlines_Dataset.json",
    orient="records",
    lines=True,
)

# 保存为普通 records 数组
news.to_json("./data/news.json", orient="records")

# 保存为 JSON Lines，每行一条记录
news.to_json(
    "./data/news-lines.json",
    orient="records",
    lines=True,
)
```

## 四、DataFrame 增删改查

### 4.1 增加列：直接赋值与 `assign`

直接赋值会修改原对象；`assign` 返回一个新 DataFrame，适合链式处理。新列若是列表，长度必须与行数一致；若是 Series，则按索引对齐。

```python
df2 = df.head().copy()

# 直接赋值：修改 df2
df2["constant"] = 33
df2["double_open"] = df2["open"] * 2

# assign：返回新对象，df2 本身不变
result = df2.assign(
    flag=1,
    spread=lambda frame: frame["close"] - frame["open"],
)
```

### 4.2 删除行、列与重复数据

`drop` 默认按行索引删除，`axis=1` 删除列；它默认返回新对象。`del` 直接删除原对象中的列，使用前应确认不再需要该列。

```python
# 删除行
without_rows = df2.drop(index=[0, 2])

# 删除列
without_column = df2.drop(columns=["constant"])

# 直接修改原对象，慎用
del df2["double_open"]

# 去除完全重复的行
deduplicated = df2.drop_duplicates()

# Series 去重
unique_values = df2["country"].drop_duplicates()
values_array = df2["country"].unique()
```

### 4.3 修改值：`assign`、赋值与 `replace`

修改前先区分“是否需要保留原始数据”。`replace` 按单元格值进行匹配替换，可以作用于 Series 或 DataFrame。

```python
# 返回修改后的副本
changed = df2.assign(GDP=66)

# 按条件修改指定位置
df2.loc[df2["GDP"] < 0, "GDP"] = 0

# 全值替换；默认返回副本
renamed = df2["country"].replace("日本", "扶桑")

# inplace=True 会修改原对象
df2["country"].replace("日本", "扶桑", inplace=True)
```

### 4.4 查询、排序、排名与聚合

常用查询方式包括 `head`、`tail`、列选择、切片、`query` 和布尔索引。排序可按一列或多列完成；`rank` 可将数值转换为名次。

```python
# 查看前后若干行
df.head(10)
df.tail(5)

# 选择一列或多列
df["country"]
df[["country", "GDP"]]

# 多列排序
df.sort_values(["year", "GDP"], ascending=[True, False])

# 分数降序排名；dense 让相同名次后不跳号
df["GDP_rank"] = df["GDP"].rank(
    method="dense",
    ascending=False,
)

# 常见聚合
df["GDP"].min()
df["GDP"].max()
df["GDP"].mean()
df["GDP"].sum()
df["GDP"].count()
```

## 五、缺失值处理

缺失值处理前先确认缺失标记和业务含义。`NaN` 可以直接由 Pandas 识别；如果数据使用 `?`、`-` 等字符串，需要先替换为 `np.nan`。

### 5.1 判断缺失值

```python
import numpy as np

# 单元格级别判断
is_missing = pd.isnull(df)
not_missing = pd.notnull(df)

# 是否存在至少一个缺失值
has_missing = np.any(pd.isnull(df))

# 是否所有值都非缺失
all_present = np.all(pd.notnull(df))
```

### 5.2 删除或填充

```python
# 删除包含 NaN 的行；默认返回副本
cleaned = df.dropna(axis="rows")

# 指定列用均值填充
df["Revenue (Millions)"] = df["Revenue (Millions)"].fillna(
    df["Revenue (Millions)"].mean()
)

# 其他填充值也可以是中位数、固定值或前后值
df["score"] = df["score"].fillna(0)
```

删除适合缺失记录很少且不能可靠推断的情况；填充适合保留样本量，但均值填充会改变分布，实际项目应结合字段含义选择策略。

### 5.3 先统一缺失标记

```python
# 将问号标记统一转换为真正的缺失值
df = df.replace(to_replace="?", value=np.nan)

# 再使用 dropna 或 fillna
df = df.dropna()
```

处理流程是“识别标记 → 统一为 `NaN` → 判断数量 → 删除或填充 → 再次检查”。

## 六、数据合并：`concat` 与 `merge`

### 6.1 `concat`：按轴拼接

`concat` 更接近“把多个表沿行或列拼起来”。`axis=0` 按行拼接，`axis=1` 按列拼接；按列拼接时依靠索引对齐。

```python
# 纵向追加记录
all_rows = pd.concat([df_jan, df_feb], axis=0, ignore_index=True)

# 横向合并两组同索引数据
combined = pd.concat([features, labels], axis=1)
```

如果上下表列结构一致，优先考虑 `axis=0`；如果两个对象代表同一批样本的不同字段，才考虑 `axis=1`。

### 6.2 `merge`：按键连接

`merge` 类似 SQL JOIN，需要明确连接键 `on` 和连接方式 `how`。

| `how` | 含义 | 保留的键 |
| --- | --- | --- |
| `inner` | 内连接 | 两边键的交集 |
| `left` | 左连接 | 左表全部键 |
| `right` | 右连接 | 右表全部键 |
| `outer` | 外连接 | 两边键的并集 |

```python
result = pd.merge(
    left,
    right,
    how="inner",
    on=["key1", "key2"],
)

left_result = pd.merge(
    left, right, how="left", on=["key1", "key2"]
)
```

连接前检查键是否唯一、两边键的数据类型是否一致。多对多连接会产生行数膨胀，必须先确认这是不是预期结果。

## 七、分组聚合与过滤

### 7.1 `groupby` 建立分组

`groupby` 返回分组对象，真正的计算要在其后调用聚合、转换或过滤方法。可以按一列或多列分组。

```python
# 按性别分组
gender_groups = df.groupby("gender_group")

# 按性别和城市分组
groups = df.groupby(["gender_group", "city"])

# 查看某一组
one_group = df.groupby(["gender_group", "channel"]).get_group(
    ("Female", "线上")
)
```

### 7.2 分组聚合 `agg`

`agg` 可以为不同字段指定不同的聚合函数，适合同时输出多种业务指标。

```python
summary = (
    df.groupby(["city", "channel"])
      .agg({
          "revenue": "mean",
          "unit_cost": "sum",
          "order": "count",
      })
)
```

分组结果的索引通常是分组键组成的 MultiIndex。如需恢复为普通列，可使用 `reset_index()`。

### 7.3 分组过滤 `filter`

`filter` 保留满足条件的整个分组，而不是只保留满足条件的单行。

```python
# 保留销售金额平均值大于 200 的城市全部记录
large_city = df.groupby("city").filter(
    lambda group: group["revenue"].mean() > 200
)

# 只对分组后的 Series 判断
large_city = df.groupby("city")["revenue"].filter(
    lambda values: values.mean() > 200
)
```

## 八、交叉表与透视表

### 8.1 交叉表 `crosstab`

交叉表用于统计两个分类变量的频数关系。例如统计不同性别中“购买/未购买”的人数。

```python
example = pd.DataFrame({
    "性别": ["男", "女", "男", "女", "男", "女"],
    "购买": ["是", "否", "是", "是", "否", "否"],
})

count_table = pd.crosstab(
    example["性别"],
    example["购买"],
)
```

如果需要比例，可以对每一行求和后再做除法：

```python
row_total = count_table.sum(axis=1).astype("float32")
proportion = count_table.div(row_total, axis=0)
```

### 8.2 透视表 `pivot_table`

透视表把字段放入行索引和列索引，再对指定值字段执行聚合。它适合快速回答“按 A 和 B 分组后，C 的平均值或总和是多少”。

```python
pivot = pd.pivot_table(
    example.assign(金额=[100, 150, 200, 130, 160, 120]),
    values="金额",
    index="性别",
    columns="购买",
    aggfunc="mean",
)
```

`crosstab` 更偏向分类频数统计；`pivot_table` 更适合对数值字段进行聚合。

### 8.3 股票涨跌与星期的关系

课程案例的思路是：先把日期转换为星期编号，再将涨跌幅转换为 0/1 分类，最后使用交叉表统计比例。

```python
# 日期索引转换为星期编号：周一为 0，周日为 6
data["week"] = pd.to_datetime(data.index).weekday

# 涨跌幅大于 0 记为 1，否则记为 0
data["posi_neg"] = np.where(data["p_change"] > 0, 1, 0)

count = pd.crosstab(data["week"], data["posi_neg"])
proportion = count.div(count.sum(axis=1), axis=0)

proportion.plot(kind="bar", stacked=True)
```

这类分析要特别注意样本量、时间范围和分类规则，交叉表只能描述样本中的关联，不能直接证明因果关系。

## 九、综合复习清单

- 算术运算：理解按索引对齐，熟悉 `add`、`sub`。
- 条件筛选：掌握布尔索引、`query`、`isin`，组合条件使用 `&` 和 `|`。
- 统计分析：会用 `describe`、聚合函数、`idxmax`/`idxmin` 和累计函数。
- 自定义处理：知道 `apply` 的 `axis=0/1` 差异，能判断是否可以用向量化替代。
- 文件读写：区分 CSV/TSV、SQL 和 JSON 的参数，注意索引与编码。
- 增删改查：区分返回副本的 `drop`/`assign` 与直接修改原对象的操作。
- 缺失值：先识别标记，再统一为 `NaN`，最后选择删除或填充。
- 合并：`concat` 按轴拼接，`merge` 按键连接；连接前检查键和基数。
- 分组：`groupby` 后使用 `agg` 聚合，使用 `filter` 保留整个分组。
- 汇总：分类频数用 `crosstab`，数值聚合用 `pivot_table`。

## 十、常见问题

### `inplace=True` 是否应该经常使用？

学习时可以使用，但在较长的数据处理流程中更推荐接收返回值。这样每一步的结果更明确，也更容易撤销或调试。

### 为什么合并后行数突然变多？

通常是连接键在一侧或两侧存在重复，形成了一对多或多对多匹配。合并前先用 `value_counts()` 检查键的唯一性。

### 为什么按时间累计的结果不符合预期？

累计函数依赖当前行顺序。先将日期转换为日期类型并按日期排序，再调用 `cumsum`、`cummax` 等函数。

### 为什么 `fillna(mean)` 可能不合适？

均值填充会压缩方差，且类别字段不能直接求均值。应结合字段类型和业务含义选择固定值、中位数、分组统计值或删除记录。

## 本篇小结

Pandas 进阶操作围绕“从数据到结论”的完整链路展开：先读入和检查数据，再完成筛选、统计、清洗和合并，最后通过分组、交叉表或透视表形成分析结果。掌握这些 API 后，面对新的表格数据时，应先画出数据关系和统计口径，再选择最小、最清晰的 Pandas 操作组合。